# Diabetes — XAI extension

The reusable XAI framework is applied to the pre-specified Random Forest with 300 trees using Glucose, BMI, Age, Insulin and SkinThickness. Median imputation is fitted on the training data, with 230 observations retained for held-out evaluation.

The framework generates explanations and assesses retraining-based stability, using accuracy for scoring. Results are exported to `../../results/diabetes`.

## Minimal model and data preparation

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

data_path = "../../data/diabetes/diabetes.csv"
df = pd.read_csv(data_path)
zero_invalid = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]


In [2]:
full_cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
             "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]

cols_3 = ["Glucose", "BMI", "Age"]
cols_5 = ["Glucose", "BMI", "Age", "Insulin", "SkinThickness"]

In [3]:
# Predictive branch: split before learning median-imputation values.
X = df[full_cols].astype(float).copy()
X[zero_invalid] = X[zero_invalid].replace(0, np.nan)
y = df["Outcome"].copy()

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X, y, test_size=230, stratify=y, random_state=42
)

predictive_imputer = SimpleImputer(strategy="median")
X_train_full.loc[:, zero_invalid] = predictive_imputer.fit_transform(
    X_train_full[zero_invalid]
)
X_test_full.loc[:, zero_invalid] = predictive_imputer.transform(
    X_test_full[zero_invalid]
)

X_train_3 = X_train_full[cols_3]
X_test_3 = X_test_full[cols_3]

X_train_5 = X_train_full[cols_5]
X_test_5 = X_test_full[cols_5]

print(X_train_full.shape, X_test_full.shape)
print("\nTrain class counts:")
print(y_train.value_counts())
print("\nTest class counts:")
print(y_test.value_counts())


(538, 8) (230, 8)

Train class counts:
Outcome
0    350
1    188
Name: count, dtype: int64

Test class counts:
Outcome
0    150
1     80
Name: count, dtype: int64


In [4]:
dataset_map = {
    "Full (8 features)": (X_train_full, X_test_full),
    "3-factor": (X_train_3, X_test_3),
    "5-factor": (X_train_5, X_test_5)
}


In [5]:
def create_model(seed=42):
    if best_model_name != "Random Forest":
        raise ValueError(f"This extension is pre-specified for Random Forest: {best_model_name}")
    return RandomForestClassifier(n_estimators=300, random_state=seed)


In [6]:
# ==========================================
# SELECT FINAL DIABETES MODEL
# ==========================================

# Pre-specified extension configuration; held-out test metrics above are
# reported descriptively and are not used for model selection.
best_model_name = "Random Forest"
best_dataset_name = "5-factor"
print("Pre-specified XAI model:", best_model_name)
print("Pre-specified feature set:", best_dataset_name)

X_train_xai, X_test_xai = dataset_map[best_dataset_name]


# Fit final model
best_model = create_model(seed=42)
best_model.fit(X_train_xai, y_train)


# Stability retraining
def train_model_fn(seed):

    model = create_model(seed)

    model.fit(X_train_xai, y_train)

    return model

Pre-specified XAI model: Random Forest
Pre-specified feature set: 5-factor


## Call the reusable XAI pipeline

In [7]:
# ==========================================
# XAI CONFIG
# ==========================================
import sys
from pathlib import Path

FRAMEWORK_DIR = Path.cwd().parent / "xai_framework"

if str(FRAMEWORK_DIR) not in sys.path:
    sys.path.insert(0, str(FRAMEWORK_DIR))

from run_pipeline import run_pipeline


config = {
    "dataset_name": "diabetes",

    "model": best_model,

    "X_train": X_train_xai,
    "X_test": X_test_xai,

    "y_train": y_train,
    "y_test": y_test,

    "feature_names": list(X_train_xai.columns),

    "train_model_fn": train_model_fn,

    "scoring": "accuracy",

    "output_dir": "../../results/diabetes"
}

experiment = run_pipeline(config)

c:\Users\catia\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Results exported to: ../../results/diabetes


## XAI framework output sanity checks

In [8]:
print("Dataset:", config["dataset_name"])
print("Model:", type(config["model"]).__name__)

print("X_train shape:", config["X_train"].shape)
print("X_test shape:", config["X_test"].shape)

print("y_train length:", len(config["y_train"]))
print("y_test length:", len(config["y_test"]))

print("Feature names:", len(config["feature_names"]))

Dataset: diabetes
Model: RandomForestClassifier
X_train shape: (538, 5)
X_test shape: (230, 5)
y_train length: 538
y_test length: 230
Feature names: 5


#### Consistency

In [9]:
assert config["X_train"].shape[1] == len(config["feature_names"])
assert config["X_test"].shape[1] == len(config["feature_names"])

assert len(config["X_train"]) == len(config["y_train"])
assert len(config["X_test"]) == len(config["y_test"])

print("Basic shape validation: OK")

Basic shape validation: OK


#### Fitted model predicts on the configured data

In [10]:
model = config["model"]
X_test = config["X_test"]

if hasattr(model, "predict_proba"):
    probs = model.predict_proba(X_test)[:, 1]
    print("Probability range:", probs.min(), probs.max())

preds = model.predict(X_test)

print("Predictions:", np.unique(preds, return_counts=True))
print("Prediction test: OK")

Probability range: 0.0 0.98
Predictions: (array([0, 1]), array([157,  73]))
Prediction test: OK


#### Results from Framework

In [11]:
global_results = experiment.state.results["global"]
local_results = experiment.state.results["local"]
stability_results = experiment.state.results["stability"]

print(global_results.keys())
print(local_results.keys())

if stability_results is not None:
    print(stability_results.keys())
else:
    print("Stability: not available")

dict_keys(['native_importance', 'permutation_importance', 'shap_importance', 'shap_values', 'shap_raw_values', 'explainer', 'rank_compare', 'rank_agreement'])
dict_keys(['cases', 'lime'])
dict_keys(['raw', 'summary', 'rank_stability'])


#### Global XAI

In [12]:
native_df = global_results["native_importance"]
perm_df = global_results["permutation_importance"]
shap_df = global_results["shap_importance"]

print("Native:")
display(native_df.head())

print("Permutation:")
display(perm_df.head())

print("SHAP:")
display(shap_df.head())

Native:


,Feature,Importance,Method
0,Glucose,0.353578,tree_importance
1,BMI,0.245440,tree_importance
2,Age,0.183218,tree_importance
3,Insulin,0.118572,tree_importance
4,SkinThickness,0.099192,tree_importance


Permutation:


,Feature,Importance,Std
0,Glucose,0.078043,0.022402
1,BMI,0.039348,0.017199
2,Age,0.026739,0.011193
3,Insulin,0.019348,0.015523
4,SkinThickness,0.010652,0.010423


SHAP:


,Feature,MeanAbsSHAP,MeanSHAP
0,Glucose,0.145322,0.001546
1,BMI,0.087032,-0.013312
2,Age,0.069504,0.001925
3,Insulin,0.037430,-0.003399
4,SkinThickness,0.022289,0.003919


#### Feature coverage

In [13]:
if native_df is not None:
    print("Native features:", len(native_df))

print("Permutation features:", len(perm_df))
print("SHAP features:", len(shap_df))
print("Expected features:", len(config["feature_names"]))

Native features: 5
Permutation features: 5
SHAP features: 5
Expected features: 5


#### Broken values

In [14]:
print("Permutation NaNs:", perm_df.isna().sum().sum())
print("SHAP NaNs:", shap_df.isna().sum().sum())

if native_df is not None:
    print("Native NaNs:", native_df.isna().sum().sum())

Permutation NaNs: 0
SHAP NaNs: 0
Native NaNs: 0


#### Rank agreement

In [15]:
print(global_results["rank_agreement"])

display(global_results["rank_compare"].head())

{'Native_vs_SHAP': np.float64(0.9999999999999999), 'Native_vs_Permutation': np.float64(0.9999999999999999), 'SHAP_vs_Permutation': np.float64(0.9999999999999999)}


,Feature,Importance_x,Method,NativeRank,MeanAbsSHAP,MeanSHAP,SHAPRank,Importance_y,Std,PermutationRank
0,Glucose,0.353578,tree_importance,1.0,0.145322,0.001546,1.0,0.078043,0.022402,1.0
1,BMI,0.245440,tree_importance,2.0,0.087032,-0.013312,2.0,0.039348,0.017199,2.0
2,Age,0.183218,tree_importance,3.0,0.069504,0.001925,3.0,0.026739,0.011193,3.0
3,Insulin,0.118572,tree_importance,4.0,0.037430,-0.003399,4.0,0.019348,0.015523,4.0
4,SkinThickness,0.099192,tree_importance,5.0,0.022289,0.003919,5.0,0.010652,0.010423,5.0


#### Representative cases

In [16]:
cases = local_results["cases"]

display(
    cases[
        ["CaseType", "y_true", "y_pred", "y_prob"]
    ]
)

,CaseType,y_true,y_pred,y_prob
52,TP,1,1,0.980000
0,TN,0,0,0.000000
24,FP,0,1,0.903333
220,FN,1,0,0.016667
17,Borderline,0,1,0.506667


#### LIME

In [17]:
lime_df = local_results["lime"]

print("LIME rows:", len(lime_df))
display(lime_df.head())

LIME rows: 25


,CaseIndex,Prediction,Probability,Rule,Weight
0,52,1,0.98,Glucose > 141.00,0.380053
1,52,1,0.98,BMI > 36.38,0.140039
2,52,1,0.98,122.00 < Insulin <= 126.00,0.044688
3,52,1,0.98,24.00 < Age <= 30.00,-0.035226
4,52,1,0.98,25.25 < SkinThickness <= 29.00,-0.029793


#### Stability

In [18]:
if stability_results is not None:
    display(stability_results["raw"].head())
    print(stability_results["summary"])

,Feature,Importance,ImportanceStd,Seed,Rank
0,Glucose,0.066957,0.029437,1,1.0
1,BMI,0.033478,0.006464,1,2.0
2,Age,0.021739,0.017821,1,4.0
3,Insulin,0.022174,0.013673,1,3.0
4,SkinThickness,0.016957,0.012964,1,5.0


         Feature  MeanImportance  StdImportance  MeanRank   StdRank
2        Glucose        0.067304       0.007445       1.0  0.000000
1            BMI        0.034870       0.011125       2.0  0.000000
0            Age        0.022783       0.009955       3.4  0.547723
3        Insulin        0.019739       0.012856       3.6  0.547723
4  SkinThickness        0.007217       0.007842       5.0  0.000000


#### Exported folder

In [19]:
from pathlib import Path

out_dir = Path(config["output_dir"]).resolve()

print("Output directory:", out_dir)

for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(out_dir))

Output directory: C:\Users\catia\Desktop\Blackboard\AASB\github\thesis\results\diabetes
bundle\metadata.json
bundle\model.pkl
bundle\X_test.csv
bundle\X_train.csv
bundle\y_test.csv
bundle\y_train.csv
lime_explanations.csv
native_importance.csv
permutation_importance.csv
rank_agreement\Native_vs_Permutation.json
rank_agreement\Native_vs_SHAP.json
rank_agreement\SHAP_vs_Permutation.json
rank_compare.csv
representative_cases.csv
shap_importance.csv
stability.csv
stability_rank.csv
stability_summary.csv


In [20]:
print(
    experiment.state.results["local"]["cases"].columns.tolist()
)

display(
    experiment.state.results["local"]["cases"]
)

['Glucose', 'BMI', 'Age', 'Insulin', 'SkinThickness', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex']


,Glucose,BMI,Age,Insulin,SkinThickness,y_true,y_pred,y_prob,distance,CaseType,CaseIndex
52,171.0,43.6,26.0,126.0,29.0,1,1,0.980000,0.480000,TP,52
0,73.0,21.1,25.0,126.0,29.0,0,0,0.000000,0.500000,TN,0
24,165.0,47.9,26.0,255.0,43.0,0,1,0.903333,0.403333,FP,24
220,106.0,26.5,29.0,126.0,24.0,1,0,0.016667,0.483333,FN,220
17,135.0,40.6,26.0,145.0,46.0,0,1,0.506667,0.006667,Borderline,17


In [21]:
assert "CaseIndex" in experiment.state.results["local"]["cases"].columns
print("Diabetes CaseIndex exists in memory.")

Diabetes CaseIndex exists in memory.


In [22]:
diabetes_cases = pd.read_csv(
    Path("../../results/diabetes/representative_cases.csv")
)

print(diabetes_cases.columns.tolist())
display(diabetes_cases)

['Glucose', 'BMI', 'Age', 'Insulin', 'SkinThickness', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex']


,Glucose,BMI,Age,Insulin,SkinThickness,y_true,y_pred,y_prob,distance,CaseType,CaseIndex
0,171.0,43.6,26.0,126.0,29.0,1,1,0.980000,0.480000,TP,52
1,73.0,21.1,25.0,126.0,29.0,0,0,0.000000,0.500000,TN,0
2,165.0,47.9,26.0,255.0,43.0,0,1,0.903333,0.403333,FP,24
3,106.0,26.5,29.0,126.0,24.0,1,0,0.016667,0.483333,FN,220
4,135.0,40.6,26.0,145.0,46.0,0,1,0.506667,0.006667,Borderline,17
